# 02b — Perbaikan Representasi Topik & Data Final untuk Dashboard
### Dashboard Identifikasi dan Analisis Tren Isu Kesehatan Masyarakat di Bangkalan

**Tujuan notebook ini:** memperbaiki representasi kata topik hasil `02_sbert_bertopic.ipynb`
secara **otomatis** (tanpa anotasi manual apa pun) dan menyiapkan seluruh dataset final yang
akan dibaca langsung oleh dashboard Streamlit.

**Yang TIDAK dilakukan notebook ini (sesuai permintaan):**
- Tidak ada file Excel untuk diisi manual, tidak ada anotasi/label manual per topik.
- Tidak mengulang cleaning (`data/raw/`, `data/cleaned/` tidak disentuh).
- Tidak menghitung ulang embedding SBERT jika `.npy` sudah ada (memuat dari cache, error
  jelas jika belum ada -- bukan menghitung ulang diam-diam).
- Tidak menjalankan BERTopic berulang kali / grid search -- model dibangun **sekali** per
  corpus, memakai konfigurasi final yang **sudah dipilih otomatis** oleh
  `02_sbert_bertopic.ipynb` (dibaca dari `final_model_selection.csv`, bukan dipilih manual
  ulang di sini).
- Tidak membuat dashboard Streamlit di notebook ini -- hanya menyiapkan datanya.

**Yang dilakukan:**
1. Memuat ulang embedding (cache), dataset modeling, dan artefak notebook 02.
2. Merekonstruksi model BERTopic per corpus dengan konfigurasi final (UMAP + HDBSCAN yang
   sama seperti saat dipilih), **tanpa mengubah cluster/topic assignment**.
3. Memperbaiki representasi kata topik dengan `CountVectorizer` + stopword Bahasa Indonesia
   + bigram, **hanya untuk ekstraksi kata representasi topik** -- teks yang dipakai SBERT sama
   sekali tidak diubah.
4. Membuat `topic_label_auto` secara otomatis dari top words (label interpretasi otomatis,
   **bukan ground truth**).
5. Menyusun seluruh dataset final: `document_topic_*.csv`, `topic_summary_*_final.csv`,
   `topic_temporal_*_final.csv`, `topic_trends_final.csv`, `dashboard_topic_data.csv`.
6. Menjalankan pengecekan konsistensi otomatis (`assert`) dan menyimpan
   `final_topic_config.json`.
7. Menampilkan checklist akhir dan status kesiapan untuk Streamlit.


## 1. Import Library & Konfigurasi

**Input:** tidak ada. **Output:** namespace siap pakai. Library sama seperti notebook 02
(BERTopic, HDBSCAN, UMAP) -- **tidak** memuat ulang `SentenceTransformer` karena embedding
sudah tersedia dari cache dan tidak akan dihitung ulang di notebook ini.


In [ ]:
!pip install -q bertopic hdbscan umap-learn Sastrawi


In [ ]:
import sys
import os
import json as json_lib
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer

import bertopic
from bertopic import BERTopic

import hdbscan as hdbscan_lib
from hdbscan import HDBSCAN

import umap as umap_lib
from umap import UMAP

import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

# Stopword Bahasa Indonesia HANYA dipakai untuk ekstraksi kata representasi topik
# (lihat Bagian 4) -- TIDAK PERNAH dipakai untuk mengubah teks input SBERT.
try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    STOPWORDS_INDONESIA = sorted(StopWordRemoverFactory().get_stop_words())
    print(f"Stopword Bahasa Indonesia dimuat dari Sastrawi ({len(STOPWORDS_INDONESIA)} kata).")
except ImportError:
    # Fallback: daftar kata umum Bahasa Indonesia yang tidak membawa makna topik.
    STOPWORDS_INDONESIA = sorted(set([
        "ada","adalah","adanya","agar","akan","akhir","akhirnya","amat","antara","antaranya",
        "apa","apabila","atas","atau","bagai","bagaimana","bagi","bahkan","bahwa","banyak",
        "bawah","beberapa","begini","begitu","belum","berada","berbagai","bermacam","bersama",
        "bertanya","betapa","bila","bisa","boleh","buat","bukan","bukankah","cara","dahulu",
        "dalam","dan","dapat","dari","daripada","dekat","demi","demikian","dengan","depan",
        "di","dia","diri","dua","engkau","guna","hal","hampir","hanya","harus","hendak",
        "hingga","ia","ialah","ini","itu","jadi","jangan","jika","juga","justru","kala",
        "kalau","kami","kamu","kapan","karena","karenanya","kata","ke","kembali","kemudian",
        "kepada","kesempatan","ketika","kini","kita","kok","lagi","lah","lain","lalu","lama",
        "langsung","lebih","maka","makanya","malah","mana","masih","masing","melainkan",
        "melalui","memang","mengapa","mereka","merupakan","meski","meskipun","mungkin",
        "nah","namun","nanti","nya","oleh","pada","padahal","paling","para","pasti","per",
        "pernah","pula","pun","sama","sambil","sampai","sana","sangat","saat","saja","saling",
        "satu","saya","se","sebab","sebagai","sebagaimana","sebelum","sebelumnya","sebetulnya",
        "sebuah","secara","sedang","sedangkan","sedikit","segala","sehingga","sejak","sekali",
        "sekarang","selain","selalu","selama","seluruh","semacam","semakin","sementara","semua",
        "semula","sendiri","seolah","seorang","sepanjang","seperti","sering","serta","sesuatu",
        "sesudah","setelah","seusai","sini","situ","suatu","sudah","supaya","tadi","tak",
        "tanpa","tapi","telah","tentang","tentu","terhadap","ternyata","tersebut","tertentu",
        "terus","tetapi","tiap","tidak","tiga","tuh","tunggu","turut","untuk","walau",
        "walaupun","yaitu","yakni","yang",
    ]))
    print(f"Sastrawi tidak tersedia -- memakai daftar stopword bawaan ({len(STOPWORDS_INDONESIA)} kata).")

print("bertopic:", bertopic.__version__ if hasattr(bertopic, "__version__") else "(lihat model_config.json)")


## 2. Load Dataset Modeling & Artefak Notebook 02

**Input:** `data/modeling/*.csv`, `data/processed/embeddings_*.npy`,
`embedding_index_*.csv`, `final_model_selection.csv`, `model_config.json` (semuanya sudah
dihasilkan `02_sbert_bertopic.ipynb`). **Output:** `df_berita`, `df_ulasan`,
`embeddings_berita`, `embeddings_ulasan`, `config_terpilih` (per corpus). Jika salah satu file
belum ada, notebook berhenti dengan pesan jelas -- **bukan** menghitung ulang secara diam-diam.


In [ ]:
# Jalankan HANYA di Google Colab. Lewati jika menjalankan secara lokal.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
BASE_DIR = "/content/drive/MyDrive/project"

MODELING_DIR = os.path.join(BASE_DIR, "data", "modeling")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
FIGURES_DIR = os.path.join(PROCESSED_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

PATHS_WAJIB = {
    "berita_modeling": os.path.join(MODELING_DIR, "berita_modeling.csv"),
    "ulasan_modeling": os.path.join(MODELING_DIR, "ulasan_modeling.csv"),
    "embeddings_berita": os.path.join(PROCESSED_DIR, "embeddings_berita.npy"),
    "embeddings_ulasan": os.path.join(PROCESSED_DIR, "embeddings_ulasan.npy"),
    "embedding_index_berita": os.path.join(PROCESSED_DIR, "embedding_index_berita.csv"),
    "embedding_index_ulasan": os.path.join(PROCESSED_DIR, "embedding_index_ulasan.csv"),
    "final_model_selection": os.path.join(PROCESSED_DIR, "final_model_selection.csv"),
    "model_config": os.path.join(PROCESSED_DIR, "model_config.json"),
}

for nama, path in PATHS_WAJIB.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File tidak ditemukan: {path}\n"
            f"Notebook ini HANYA membaca hasil 02_sbert_bertopic.ipynb -- jalankan notebook "
            f"tersebut sampai selesai sebelum menjalankan 02b."
        )

df_berita = pd.read_csv(PATHS_WAJIB["berita_modeling"], encoding="utf-8-sig")
df_ulasan = pd.read_csv(PATHS_WAJIB["ulasan_modeling"], encoding="utf-8-sig")

embeddings_berita = np.load(PATHS_WAJIB["embeddings_berita"])
embeddings_ulasan = np.load(PATHS_WAJIB["embeddings_ulasan"])

embedding_index_berita = pd.read_csv(PATHS_WAJIB["embedding_index_berita"])
embedding_index_ulasan = pd.read_csv(PATHS_WAJIB["embedding_index_ulasan"])

final_model_selection = pd.read_csv(PATHS_WAJIB["final_model_selection"])

with open(PATHS_WAJIB["model_config"], encoding="utf-8") as f:
    model_config_notebook02 = json_lib.load(f)

print("df_berita           :", df_berita.shape)
print("df_ulasan           :", df_ulasan.shape)
print("embeddings_berita    :", embeddings_berita.shape)
print("embeddings_ulasan    :", embeddings_ulasan.shape)
print("\nfinal_model_selection (dari notebook 02):")
display(final_model_selection[["dataset", "model_id", "min_topic_size", "jumlah_topic",
                                 "proporsi_outlier", "skor_composite"]])


In [ ]:
# --- Pengecekan konsistensi urutan embedding <-> document_id (WAJIB sebelum dipakai) ---
assert list(embedding_index_berita["document_id"]) == list(df_berita["document_id"]), (
    "Urutan document_id pada embedding_index_berita.csv TIDAK sama dengan berita_modeling.csv. "
    "Jangan lanjutkan -- periksa apakah dataset modeling berubah setelah embedding dihitung."
)
assert list(embedding_index_ulasan["document_id"]) == list(df_ulasan["document_id"]), (
    "Urutan document_id pada embedding_index_ulasan.csv TIDAK sama dengan ulasan_modeling.csv."
)
assert embeddings_berita.shape[0] == len(df_berita)
assert embeddings_ulasan.shape[0] == len(df_ulasan)
print("OK: urutan embedding konsisten dengan document_id pada kedua dataset modeling.")

RANDOM_STATE = model_config_notebook02["random_state"]
UMAP_CONFIG = model_config_notebook02["umap_config"]
print(f"\nRANDOM_STATE (dari notebook 02) : {RANDOM_STATE}")
print(f"UMAP_CONFIG (dari notebook 02)  : {UMAP_CONFIG}")


## 3. Rekonstruksi Model BERTopic (Konfigurasi Final)

**Tujuan:** membangun ulang model BERTopic memakai `min_cluster_size` yang **sudah dipilih
otomatis** oleh notebook 02 (`final_model_selection.csv`) -- **satu kali per corpus**, bukan
sweep beberapa konfigurasi lagi. Embedding dipakai dari cache (Bagian 2), sehingga tidak ada
komputasi SBERT sama sekali di notebook ini.
**Input:** `documents_*`, `embeddings_*`, `min_cluster_size` terpilih. **Output:**
`topic_model_berita`, `topic_model_ulasan`, `topics_*`, `probs_*`. **Dipakai di:** Bagian 4-13.


In [ ]:
TEXT_COL = "text"
documents_berita = df_berita[TEXT_COL].astype(str).tolist()
ids_berita = df_berita["document_id"].tolist()
documents_ulasan = df_ulasan[TEXT_COL].astype(str).tolist()
ids_ulasan = df_ulasan["document_id"].tolist()

# --- assert tidak ada text kosong (dipakai lagi di Bagian 12 sebagai validasi resmi) ---
assert all(len(d.strip()) > 0 for d in documents_berita), "Ditemukan text kosong pada berita!"
assert all(len(d.strip()) > 0 for d in documents_ulasan), "Ditemukan text kosong pada ulasan!"

MCS_BERITA = int(final_model_selection.loc[final_model_selection["dataset"] == "berita", "min_topic_size"].iloc[0])
MCS_ULASAN = int(final_model_selection.loc[final_model_selection["dataset"] == "ulasan", "min_topic_size"].iloc[0])
print(f"min_cluster_size terpilih -- berita: {MCS_BERITA}, ulasan: {MCS_ULASAN} "
      f"(diambil otomatis dari final_model_selection.csv, TIDAK dipilih manual di sini)")


In [ ]:
def bangun_topic_model(documents: list, embeddings: np.ndarray, min_cluster_size: int,
                        umap_config: dict, random_state: int):
    """Sama seperti pada notebook 02 -- dipakai di sini HANYA sekali per corpus
    (bukan sweep), memakai konfigurasi yang sudah final."""
    umap_model = UMAP(
        n_neighbors=umap_config["n_neighbors"], n_components=umap_config["n_components"],
        min_dist=umap_config["min_dist"], metric=umap_config["metric"],
        random_state=umap_config["random_state"],
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size, min_samples=None, metric="euclidean",
        cluster_selection_method="eom", prediction_data=True,
    )
    topic_model = BERTopic(
        embedding_model=None, umap_model=umap_model, hdbscan_model=hdbscan_model,
        calculate_probabilities=True, verbose=False,
    )
    t0 = time.time()
    topics, probs = topic_model.fit_transform(documents, embeddings=embeddings)
    durasi = time.time() - t0
    return topic_model, topics, probs, durasi


t0_total = time.time()
topic_model_berita, topics_berita, probs_berita, durasi_berita = bangun_topic_model(
    documents_berita, embeddings_berita, MCS_BERITA, UMAP_CONFIG, RANDOM_STATE
)
print(f"[berita] Model dibangun dalam {durasi_berita:.1f} detik "
      f"({(topic_model_berita.get_topic_info()['Topic'] != -1).sum()} topik reguler)")

topic_model_ulasan, topics_ulasan, probs_ulasan, durasi_ulasan = bangun_topic_model(
    documents_ulasan, embeddings_ulasan, MCS_ULASAN, UMAP_CONFIG, RANDOM_STATE
)
print(f"[ulasan] Model dibangun dalam {durasi_ulasan:.1f} detik "
      f"({(topic_model_ulasan.get_topic_info()['Topic'] != -1).sum()} topik reguler)")


## 4. Perbaiki Representasi Kata Topik

**Tujuan:** mengganti kata-kata representasi topik (top words) yang tadinya masih memuat
kata umum ("dan", "yang", "di", dst.) dengan representasi yang lebih informatif, memakai
`CountVectorizer` dengan stopword Bahasa Indonesia dan bigram. **Ini HANYA mengubah cara
kata representasi topik diekstrak (c-TF-IDF BERTopic) -- clustering, embedding, dan teks asli
SAMA SEKALI TIDAK berubah** (`update_topics()` BERTopic tidak menghitung ulang UMAP/HDBSCAN).
**Input:** `topic_model_berita`, `topic_model_ulasan`, `documents_*`. **Output:** model yang
sama dengan representasi kata yang sudah diperbarui. **Dipakai di:** Bagian 5-7.

Kata "Bangkalan" (atau nama wilayah lain) **tidak** dihapus secara otomatis di sini -- hanya
stopword fungsi (STOPWORDS_INDONESIA) yang disaring, bukan nama tempat.


In [ ]:
vectorizer_representasi = CountVectorizer(
    stop_words=STOPWORDS_INDONESIA,
    ngram_range=(1, 2),
    min_df=2,
)

topic_model_berita.update_topics(documents_berita, vectorizer_model=vectorizer_representasi)
topic_model_ulasan.update_topics(documents_ulasan, vectorizer_model=vectorizer_representasi)

print("Representasi kata topik diperbarui (stopword ID + unigram/bigram, min_df=2).")
print("\nContoh top words berita SETELAH perbaikan (5 topik pertama, tanpa outlier):")
for tid in [t for t in topic_model_berita.get_topics().keys() if t != -1][:5]:
    kata = [k for k, s in topic_model_berita.get_topic(tid)[:8]]
    print(f"  Topic {tid}: {', '.join(kata)}")


## 5. Label Topik Otomatis (`topic_label_auto`)

**Tujuan:** membuat label ringkas otomatis per topik dari top words -- **interpretasi
otomatis, bukan ground truth** (tidak ada validasi manusia atas label ini pada notebook ini).
**Input:** top words tiap topik (setelah Bagian 4). **Output:** dict `{topic_id: label}` per
corpus. **Dipakai di:** Bagian 6-11.

Aturan pembuatan label (heuristik sederhana & deterministik, didokumentasikan agar bisa
ditelusuri): ambil 2 kata/frasa top pertama (sudah tidak mengandung stopword), gabungkan
dengan "dan", huruf awal tiap kata dikapitalkan. Topik -1 (outlier) diberi label tetap
`"Outlier / Tidak Terklasifikasi"`, bukan tema substantif.


In [ ]:
def buat_label_topik_otomatis(topic_model: BERTopic, top_n: int = 2) -> dict:
    """
    Label sederhana & deterministik dari top-N kata/frasa representasi topik.
    Contoh: top_words=['sampah','limbah','sungai'] -> 'Sampah dan Limbah'.
    Label ini adalah interpretasi OTOMATIS -- peneliti tetap disarankan meninjau label
    ini secara kualitatif lewat representative documents sebelum dipakai di Bab IV.
    """
    label_per_topik = {}
    for tid, daftar_kata_skor in topic_model.get_topics().items():
        if tid == -1:
            label_per_topik[tid] = "Outlier / Tidak Terklasifikasi"
            continue
        kata_top = [kata for kata, skor in daftar_kata_skor[:top_n]]
        if not kata_top:
            label_per_topik[tid] = f"Topik {tid} (tanpa kata representatif)"
            continue
        kata_kapital = [k.title() for k in kata_top]
        label_per_topik[tid] = " dan ".join(kata_kapital)
    return label_per_topik


label_topik_berita = buat_label_topik_otomatis(topic_model_berita)
label_topik_ulasan = buat_label_topik_otomatis(topic_model_ulasan)

print("=== topic_label_auto -- berita ===")
for tid, label in label_topik_berita.items():
    print(f"  Topic {tid}: {label}")
print("\n=== topic_label_auto -- ulasan ===")
for tid, label in label_topik_ulasan.items():
    print(f"  Topic {tid}: {label}")


## 6. Statistik Outlier

**Tujuan:** melaporkan proporsi dokumen yang tidak masuk topik manapun (`topic -1`) secara
eksplisit -- **topic -1 TIDAK diperlakukan sebagai topik kesehatan/topik substantif apa pun.**
**Input:** `topics_berita`, `topics_ulasan`. **Output:** ringkasan dicetak + dipakai di
`final_topic_config.json` (Bagian 15).


In [ ]:
def statistik_outlier(topics: list, dataset_name: str) -> dict:
    total_dokumen = len(topics)
    total_outlier = sum(1 for t in topics if t == -1)
    total_topik_reguler = len(set(t for t in topics if t != -1))
    persentase_outlier = total_outlier / total_dokumen if total_dokumen > 0 else float("nan")

    print(f"[{dataset_name}] total_documents={total_dokumen}, "
          f"total_regular_topics={total_topik_reguler}, total_outliers={total_outlier}, "
          f"outlier_percentage={persentase_outlier:.1%}")
    return {
        "total_documents": total_dokumen,
        "total_regular_topics": total_topik_reguler,
        "total_outliers": total_outlier,
        "outlier_percentage": round(persentase_outlier, 4),
    }


statistik_outlier_berita = statistik_outlier(topics_berita, "berita")
statistik_outlier_ulasan = statistik_outlier(topics_ulasan, "ulasan")


## 7. Document-Topic Dataset

**Tujuan:** menggabungkan `document_id`, teks, topic assignment, probabilitas, label otomatis,
dan metadata waktu + metadata asli lain per corpus. **Input:** `df_berita`/`df_ulasan`,
`topics_*`, `probs_*`, `label_topik_*`. **Output:** `document_topic_berita.csv`,
`document_topic_ulasan.csv`. **Dipakai di:** Bagian 8-11.


In [ ]:
def ambil_topic_probability(probs) -> list:
    if probs is not None and hasattr(probs, "ndim") and probs.ndim == 2 and probs.shape[1] > 0:
        return probs.max(axis=1).tolist()
    return [None] * (probs.shape[0] if hasattr(probs, "shape") else 0)


def buat_document_topic_final(df: pd.DataFrame, topics: list, probs, label_topik: dict,
                               kolom_metadata_tambahan: list) -> pd.DataFrame:
    df_hasil = df[["document_id", "source", "text", "tahun", "bulan", "tahun_bulan"] +
                   kolom_metadata_tambahan].copy()
    df_hasil["topic"] = topics
    df_hasil["topic_probability"] = ambil_topic_probability(probs)
    df_hasil["topic_label_auto"] = df_hasil["topic"].map(label_topik)

    assert df_hasil["document_id"].notna().all(), "Ada document_id kosong!"
    assert df_hasil["document_id"].is_unique, "Ada document_id duplikat!"
    return df_hasil


document_topic_berita = buat_document_topic_final(
    df_berita, topics_berita, probs_berita, label_topik_berita,
    kolom_metadata_tambahan=["publisher", "kecamatan", "url"],
)
document_topic_ulasan = buat_document_topic_final(
    df_ulasan, topics_ulasan, probs_ulasan, label_topik_ulasan,
    kolom_metadata_tambahan=["place_name", "place_category", "rating"],
)

assert "reviewer_name" not in document_topic_ulasan.columns
assert "place_address" not in document_topic_ulasan.columns

DOCUMENT_TOPIC_BERITA_PATH = os.path.join(PROCESSED_DIR, "document_topic_berita.csv")
DOCUMENT_TOPIC_ULASAN_PATH = os.path.join(PROCESSED_DIR, "document_topic_ulasan.csv")
document_topic_berita.to_csv(DOCUMENT_TOPIC_BERITA_PATH, index=False, encoding="utf-8-sig")
document_topic_ulasan.to_csv(DOCUMENT_TOPIC_ULASAN_PATH, index=False, encoding="utf-8-sig")

print("Tersimpan:", DOCUMENT_TOPIC_BERITA_PATH, "|", len(document_topic_berita), "baris")
print("Tersimpan:", DOCUMENT_TOPIC_ULASAN_PATH, "|", len(document_topic_ulasan), "baris")
display(document_topic_berita.head(3))


## 8. Topic Summary Final

**Tujuan:** ringkasan akhir tiap topik (ukuran, persentase, kata representatif setelah
perbaikan Bagian 4, label otomatis, contoh dokumen). **Input:** `topic_model_*`,
`document_topic_*`. **Output:** `topic_summary_berita_final.csv`,
`topic_summary_ulasan_final.csv`. **Dipakai di:** Bagian 11 (dashboard_topic_data.csv), Bab IV.


In [ ]:
def buat_topic_summary_final(topic_model: BERTopic, topics: list, documents: list, ids: list,
                              label_topik: dict, top_n_kata: int = 10,
                              top_n_dokumen: int = 3) -> pd.DataFrame:
    topic_info = topic_model.get_topic_info()
    total_dokumen = len(topics)
    doc_id_by_pos = {i: doc_id for i, doc_id in enumerate(ids)}

    baris = []
    for _, row in topic_info.iterrows():
        tid = row["Topic"]
        ukuran = row["Count"]

        if tid == -1:
            top_words = "(outlier -- tidak ada representasi kata topik)"
        else:
            top_words = ", ".join([kata for kata, skor in topic_model.get_topic(tid)[:top_n_kata]])

        posisi = [i for i, t in enumerate(topics) if t == tid][:top_n_dokumen]
        rep_ids = [doc_id_by_pos[i] for i in posisi]

        baris.append({
            "topic": tid,
            "topic_label_auto": label_topik.get(tid, ""),
            "topic_size": ukuran,
            "percentage": round(ukuran / total_dokumen, 4) if total_dokumen > 0 else None,
            "top_words": top_words,
            "representative_document_ids": "; ".join(rep_ids),
        })
    return pd.DataFrame(baris)


topic_summary_berita_final = buat_topic_summary_final(
    topic_model_berita, topics_berita, documents_berita, ids_berita, label_topik_berita
)
topic_summary_ulasan_final = buat_topic_summary_final(
    topic_model_ulasan, topics_ulasan, documents_ulasan, ids_ulasan, label_topik_ulasan
)

TOPIC_SUMMARY_BERITA_FINAL_PATH = os.path.join(PROCESSED_DIR, "topic_summary_berita_final.csv")
TOPIC_SUMMARY_ULASAN_FINAL_PATH = os.path.join(PROCESSED_DIR, "topic_summary_ulasan_final.csv")
topic_summary_berita_final.to_csv(TOPIC_SUMMARY_BERITA_FINAL_PATH, index=False, encoding="utf-8-sig")
topic_summary_ulasan_final.to_csv(TOPIC_SUMMARY_ULASAN_FINAL_PATH, index=False, encoding="utf-8-sig")

print("Tersimpan:", TOPIC_SUMMARY_BERITA_FINAL_PATH)
print("Tersimpan:", TOPIC_SUMMARY_ULASAN_FINAL_PATH)
display(topic_summary_berita_final)


## 9. Analisis Temporal Final & Topic-Trend Table

**Tujuan:** menghitung jumlah dokumen dan **proporsi** tiap topik per `tahun_bulan` (agar
periode dengan jumlah dokumen berbeda tetap bisa dibandingkan secara adil), lalu menyusun satu
tabel gabungan ringkas (`topic_trends_final.csv`) untuk grafik tren di Streamlit.
**Input:** `document_topic_*`. **Output:** `topic_temporal_berita_final.csv`,
`topic_temporal_ulasan_final.csv`, `topic_trends_final.csv`. **Dipakai di:** Bagian 10-11.


In [ ]:
def buat_topic_temporal_final(df_topic: pd.DataFrame) -> pd.DataFrame:
    agg = (
        df_topic.groupby(["tahun_bulan", "topic"]).size()
        .rename("jumlah_dokumen").reset_index()
    )
    total_per_periode = df_topic.groupby("tahun_bulan").size().rename("total_dokumen_periode")
    agg = agg.merge(total_per_periode, on="tahun_bulan")
    agg["proporsi"] = (agg["jumlah_dokumen"] / agg["total_dokumen_periode"]).round(4)

    # tahun & bulan diturunkan dari tahun_bulan (format YYYY-MM) untuk kemudahan filter Streamlit
    tanggal_parsed = pd.to_datetime(agg["tahun_bulan"], format="%Y-%m", errors="coerce")
    agg.insert(1, "tahun", tanggal_parsed.dt.year)
    agg.insert(2, "bulan", tanggal_parsed.dt.month)

    return agg[["topic", "tahun", "bulan", "tahun_bulan", "jumlah_dokumen", "proporsi"]]


topic_temporal_berita_final = buat_topic_temporal_final(document_topic_berita)
topic_temporal_ulasan_final = buat_topic_temporal_final(document_topic_ulasan)

TOPIC_TEMPORAL_BERITA_FINAL_PATH = os.path.join(PROCESSED_DIR, "topic_temporal_berita_final.csv")
TOPIC_TEMPORAL_ULASAN_FINAL_PATH = os.path.join(PROCESSED_DIR, "topic_temporal_ulasan_final.csv")
topic_temporal_berita_final.to_csv(TOPIC_TEMPORAL_BERITA_FINAL_PATH, index=False, encoding="utf-8-sig")
topic_temporal_ulasan_final.to_csv(TOPIC_TEMPORAL_ULASAN_FINAL_PATH, index=False, encoding="utf-8-sig")

print("Tersimpan:", TOPIC_TEMPORAL_BERITA_FINAL_PATH, "|", len(topic_temporal_berita_final), "baris")
print("Tersimpan:", TOPIC_TEMPORAL_ULASAN_FINAL_PATH, "|", len(topic_temporal_ulasan_final), "baris")


In [ ]:
def tambahkan_dataset_dan_label(df_temporal: pd.DataFrame, dataset_name: str, label_topik: dict) -> pd.DataFrame:
    df = df_temporal.copy()
    df.insert(0, "dataset", dataset_name)
    df.insert(2, "topic_label_auto", df["topic"].map(label_topik))
    return df[["dataset", "topic", "topic_label_auto", "tahun_bulan", "jumlah_dokumen", "proporsi"]]


topic_trends_final = pd.concat([
    tambahkan_dataset_dan_label(topic_temporal_berita_final, "berita", label_topik_berita),
    tambahkan_dataset_dan_label(topic_temporal_ulasan_final, "ulasan", label_topik_ulasan),
], ignore_index=True)

TOPIC_TRENDS_FINAL_PATH = os.path.join(PROCESSED_DIR, "topic_trends_final.csv")
topic_trends_final.to_csv(TOPIC_TRENDS_FINAL_PATH, index=False, encoding="utf-8-sig")

print("Tersimpan:", TOPIC_TRENDS_FINAL_PATH, "|", len(topic_trends_final), "baris")
display(topic_trends_final.head(10))


## 10. Visualisasi

**Tujuan:** 4 grafik ringkas (bukan eksplorasi menyeluruh): distribusi ukuran topik, distribusi
outlier, tren topik terhadap waktu, dan top words tiap topik (setelah perbaikan Bagian 4).
**Input:** `topic_summary_*_final`, `statistik_outlier_*`, `topic_temporal_*_final`.
**Output:** file `.png` di `data/processed/figures/`.


In [ ]:
def plot_distribusi_topic_size_final(topic_summary: pd.DataFrame, dataset_name: str):
    reguler = topic_summary[topic_summary["topic"] != -1].sort_values("topic_size", ascending=False)
    if reguler.empty:
        print(f"[{dataset_name}] Tidak ada topik reguler -- grafik dilewati.")
        return
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(reguler))))
    ax.barh(reguler["topic_label_auto"], reguler["topic_size"])
    ax.set_xlabel("Jumlah dokumen")
    ax.set_title(f"Distribusi ukuran topik -- {dataset_name}")
    ax.invert_yaxis()
    fig.tight_layout()
    path = os.path.join(FIGURES_DIR, f"final_distribusi_topic_size_{dataset_name}.png")
    fig.savefig(path, dpi=120); plt.close(fig)
    print("Tersimpan:", path)


def plot_distribusi_outlier_final(stat: dict, dataset_name: str):
    labels = ["Topik reguler", "Outlier (-1)"]
    values = [stat["total_documents"] - stat["total_outliers"], stat["total_outliers"]]
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.pie(values, labels=labels, autopct="%1.1f%%")
    ax.set_title(f"Proporsi outlier -- {dataset_name}")
    fig.tight_layout()
    path = os.path.join(FIGURES_DIR, f"final_distribusi_outlier_{dataset_name}.png")
    fig.savefig(path, dpi=120); plt.close(fig)
    print("Tersimpan:", path)


def plot_tren_temporal_final(topic_temporal: pd.DataFrame, label_topik: dict, dataset_name: str,
                              top_n_topik: int = 5):
    reguler = topic_temporal[topic_temporal["topic"] != -1]
    if reguler.empty:
        print(f"[{dataset_name}] Tidak ada topik reguler -- grafik tren dilewati.")
        return
    topik_terbesar = reguler.groupby("topic")["jumlah_dokumen"].sum().nlargest(top_n_topik).index
    fig, ax = plt.subplots(figsize=(9, 5))
    for tid in topik_terbesar:
        subset = reguler[reguler["topic"] == tid].sort_values("tahun_bulan")
        ax.plot(subset["tahun_bulan"], subset["proporsi"], marker="o",
                label=label_topik.get(tid, f"Topic {tid}"))
    ax.set_xlabel("Periode (tahun-bulan)")
    ax.set_ylabel("Proporsi topik dalam periode")
    ax.set_title(f"Tren topik dari waktu ke waktu (top {top_n_topik}) -- {dataset_name}")
    ax.legend(fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=90, fontsize=6)
    fig.tight_layout()
    path = os.path.join(FIGURES_DIR, f"final_tren_temporal_{dataset_name}.png")
    fig.savefig(path, dpi=120); plt.close(fig)
    print("Tersimpan:", path)


def plot_top_words_final(topic_model: BERTopic, label_topik: dict, dataset_name: str,
                          top_n_topik: int = 4, top_n_kata: int = 8):
    topik_reguler = [t for t in topic_model.get_topics().keys() if t != -1]
    if not topik_reguler:
        print(f"[{dataset_name}] Tidak ada topik reguler -- grafik top words dilewati.")
        return
    topik_dipilih = topik_reguler[:top_n_topik]

    fig, axes = plt.subplots(1, len(topik_dipilih), figsize=(4 * len(topik_dipilih), 4))
    if len(topik_dipilih) == 1:
        axes = [axes]
    for ax, tid in zip(axes, topik_dipilih):
        kata_skor = topic_model.get_topic(tid)[:top_n_kata]
        kata = [k for k, s in kata_skor][::-1]
        skor = [s for k, s in kata_skor][::-1]
        ax.barh(kata, skor)
        ax.set_title(label_topik.get(tid, f"Topic {tid}"), fontsize=9)
    fig.suptitle(f"Top words per topik -- {dataset_name}")
    fig.tight_layout()
    path = os.path.join(FIGURES_DIR, f"final_top_words_{dataset_name}.png")
    fig.savefig(path, dpi=120); plt.close(fig)
    print("Tersimpan:", path)


plot_distribusi_topic_size_final(topic_summary_berita_final, "berita")
plot_distribusi_topic_size_final(topic_summary_ulasan_final, "ulasan")
plot_distribusi_outlier_final(statistik_outlier_berita, "berita")
plot_distribusi_outlier_final(statistik_outlier_ulasan, "ulasan")
plot_tren_temporal_final(topic_temporal_berita_final, label_topik_berita, "berita")
plot_tren_temporal_final(topic_temporal_ulasan_final, label_topik_ulasan, "ulasan")
plot_top_words_final(topic_model_berita, label_topik_berita, "berita")
plot_top_words_final(topic_model_ulasan, label_topik_ulasan, "ulasan")


## 11. Dashboard-Ready Data

**Tujuan:** satu file konsolidasi yang bisa dibaca langsung oleh Streamlit **tanpa menjalankan
SBERT/BERTopic** -- dashboard hanya membaca file ini (dan file lain di `data/processed/` bila
perlu detail lebih lanjut). **Input:** `document_topic_*` (Bagian 7). **Output:**
`dashboard_topic_data.csv`.


In [ ]:
KOLOM_UMUM = ["document_id", "source", "topic", "topic_label_auto", "topic_probability",
              "text", "tahun", "bulan", "tahun_bulan"]
KOLOM_KHUSUS_BERITA = ["publisher", "kecamatan", "url"]
KOLOM_KHUSUS_ULASAN = ["place_name", "place_category", "rating"]

dashboard_berita = document_topic_berita[KOLOM_UMUM + KOLOM_KHUSUS_BERITA].copy()
for kolom in KOLOM_KHUSUS_ULASAN:
    dashboard_berita[kolom] = pd.NA

dashboard_ulasan = document_topic_ulasan[KOLOM_UMUM + KOLOM_KHUSUS_ULASAN].copy()
for kolom in KOLOM_KHUSUS_BERITA:
    dashboard_ulasan[kolom] = pd.NA

dashboard_topic_data = pd.concat([
    dashboard_berita[KOLOM_UMUM + KOLOM_KHUSUS_BERITA + KOLOM_KHUSUS_ULASAN],
    dashboard_ulasan[KOLOM_UMUM + KOLOM_KHUSUS_BERITA + KOLOM_KHUSUS_ULASAN],
], ignore_index=True)

assert "reviewer_name" not in dashboard_topic_data.columns
assert "place_address" not in dashboard_topic_data.columns
assert dashboard_topic_data["document_id"].is_unique

DASHBOARD_TOPIC_DATA_PATH = os.path.join(PROCESSED_DIR, "dashboard_topic_data.csv")
dashboard_topic_data.to_csv(DASHBOARD_TOPIC_DATA_PATH, index=False, encoding="utf-8-sig")

print("Tersimpan:", DASHBOARD_TOPIC_DATA_PATH, "|", len(dashboard_topic_data), "baris,",
      len(dashboard_topic_data.columns), "kolom")
display(dashboard_topic_data.head(3))


## 12. Validation Checks

**Tujuan:** memastikan seluruh output final konsisten satu sama lain sebelum dipakai
Streamlit. Semua pengecekan memakai `assert` -- notebook berhenti dengan pesan jelas jika ada
yang tidak konsisten (bukan melanjutkan dengan data yang salah).


In [ ]:
def jalankan_validation_checks(df_topic: pd.DataFrame, topic_summary: pd.DataFrame,
                                topic_temporal: pd.DataFrame, topics: list, embeddings: np.ndarray,
                                dataset_name: str):
    print(f"=== Validation checks: {dataset_name} ===")

    # 1) jumlah document_id unik & tidak ada yang hilang
    assert df_topic["document_id"].notna().all(), f"[{dataset_name}] Ada document_id kosong!"
    assert df_topic["document_id"].is_unique, f"[{dataset_name}] Ada document_id duplikat!"
    print("  [OK] document_id unik dan tidak ada yang hilang")

    # 2) jumlah topic assignment sama dengan jumlah dokumen
    assert len(topics) == len(df_topic), f"[{dataset_name}] jumlah topic assignment != jumlah dokumen!"
    print("  [OK] jumlah topic assignment == jumlah dokumen")

    # 3) jumlah embedding sesuai jumlah dokumen
    assert embeddings.shape[0] == len(df_topic), f"[{dataset_name}] jumlah embedding != jumlah dokumen!"
    print("  [OK] jumlah embedding == jumlah dokumen")

    # 4) topic summary sesuai topic assignment (ukuran topik cocok dengan hitungan langsung)
    hitung_langsung = pd.Series(topics).value_counts()
    for _, row in topic_summary.iterrows():
        tid, ukuran_tercatat = row["topic"], row["topic_size"]
        ukuran_aktual = hitung_langsung.get(tid, 0)
        assert ukuran_tercatat == ukuran_aktual, (
            f"[{dataset_name}] topic_size topik {tid} tidak cocok: "
            f"tercatat={ukuran_tercatat}, aktual={ukuran_aktual}"
        )
    print("  [OK] topic_summary konsisten dengan topic assignment")

    # 5) tidak ada text kosong
    assert (df_topic["text"].fillna("").astype(str).str.len() > 0).all(), \
        f"[{dataset_name}] Ada text kosong pada document_topic!"
    print("  [OK] tidak ada text kosong")

    # 6) topic_temporal konsisten dengan document-topic (total jumlah_dokumen per topik cocok)
    total_dari_temporal = topic_temporal.groupby("topic")["jumlah_dokumen"].sum()
    for tid, total_aktual in hitung_langsung.items():
        total_tercatat = total_dari_temporal.get(tid, 0)
        assert total_tercatat == total_aktual, (
            f"[{dataset_name}] Total jumlah_dokumen topik {tid} pada topic_temporal "
            f"({total_tercatat}) tidak cocok dengan document_topic ({total_aktual})"
        )
    print("  [OK] topic_temporal konsisten dengan document-topic")
    print(f"=== Semua pengecekan LULUS untuk {dataset_name} ===\n")


jalankan_validation_checks(document_topic_berita, topic_summary_berita_final,
                            topic_temporal_berita_final, topics_berita, embeddings_berita, "berita")
jalankan_validation_checks(document_topic_ulasan, topic_summary_ulasan_final,
                            topic_temporal_ulasan_final, topics_ulasan, embeddings_ulasan, "ulasan")


## 13. Reproducibility & Konfigurasi Final

**Tujuan:** mencatat seluruh konfigurasi yang dipakai notebook ini (model SBERT yang dipakai
notebook 02, konfigurasi BERTopic/HDBSCAN/UMAP, jumlah dokumen/topik/outlier, waktu proses)
untuk dilampirkan ke Bab III. **Output:** `final_topic_config.json`.


In [ ]:
print("=== Ringkasan reproducibility ===")
print("Model SBERT (dari notebook 02) :", model_config_notebook02["sbert"]["model_used"])
print("min_cluster_size -- berita     :", MCS_BERITA)
print("min_cluster_size -- ulasan     :", MCS_ULASAN)
print("Jumlah dokumen -- berita       :", len(df_berita))
print("Jumlah dokumen -- ulasan       :", len(df_ulasan))
print("Jumlah topik reguler -- berita :", statistik_outlier_berita['total_regular_topics'])
print("Jumlah topik reguler -- ulasan :", statistik_outlier_ulasan['total_regular_topics'])
print("Jumlah outlier -- berita       :", statistik_outlier_berita['total_outliers'])
print("Jumlah outlier -- ulasan       :", statistik_outlier_ulasan['total_outliers'])
print(f"Waktu proses model -- berita   : {durasi_berita:.1f} detik")
print(f"Waktu proses model -- ulasan   : {durasi_ulasan:.1f} detik")


In [ ]:
final_topic_config = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "sumber_konfigurasi": "diambil otomatis dari data/processed/final_model_selection.csv "
                          "dan model_config.json hasil 02_sbert_bertopic.ipynb",
    "random_state": RANDOM_STATE,
    "sbert_model_used": model_config_notebook02["sbert"]["model_used"],
    "umap_config": UMAP_CONFIG,
    "hdbscan": {
        "min_cluster_size_berita": MCS_BERITA,
        "min_cluster_size_ulasan": MCS_ULASAN,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
    },
    "representasi_topik": {
        "vectorizer": "CountVectorizer",
        "stop_words": "Indonesia (Sastrawi jika tersedia, fallback daftar bawaan)",
        "ngram_range": [1, 2],
        "min_df": 2,
        "catatan": "Hanya memengaruhi ekstraksi kata representasi topik (c-TF-IDF), "
                   "tidak mengubah teks input SBERT maupun hasil clustering.",
    },
    "hasil": {
        "berita": {**statistik_outlier_berita, "waktu_proses_detik": round(durasi_berita, 2)},
        "ulasan": {**statistik_outlier_ulasan, "waktu_proses_detik": round(durasi_ulasan, 2)},
    },
    "output_files": [
        "document_topic_berita.csv", "document_topic_ulasan.csv",
        "topic_summary_berita_final.csv", "topic_summary_ulasan_final.csv",
        "topic_temporal_berita_final.csv", "topic_temporal_ulasan_final.csv",
        "topic_trends_final.csv", "dashboard_topic_data.csv", "final_topic_config.json",
    ],
}

FINAL_TOPIC_CONFIG_PATH = os.path.join(PROCESSED_DIR, "final_topic_config.json")
with open(FINAL_TOPIC_CONFIG_PATH, "w", encoding="utf-8") as f:
    json_lib.dump(final_topic_config, f, ensure_ascii=False, indent=2)

print("Tersimpan:", FINAL_TOPIC_CONFIG_PATH)


In [ ]:
print("=== DAFTAR FILE OUTPUT data/processed/ (setelah notebook 02b) ===")
for nama_file in sorted(os.listdir(PROCESSED_DIR)):
    path_lengkap = os.path.join(PROCESSED_DIR, nama_file)
    if os.path.isfile(path_lengkap):
        ukuran_kb = os.path.getsize(path_lengkap) / 1024
        print(f"  {nama_file:<42} {ukuran_kb:>10.1f} KB")


## Checklist & Status Kesiapan Streamlit

```
[OK] topic representation final   -- top words diperbarui dengan stopword ID + bigram (Bagian 4)
[OK] document-topic dataset       -- document_topic_berita.csv, document_topic_ulasan.csv (Bagian 7)
[OK] topic summary                -- topic_summary_berita_final.csv, topic_summary_ulasan_final.csv (Bagian 8)
[OK] temporal analysis            -- topic_temporal_*_final.csv, topic_trends_final.csv (Bagian 9)
[OK] dashboard-ready data          -- dashboard_topic_data.csv (Bagian 11)
[OK] configuration saved          -- final_topic_config.json (Bagian 13)
```

**Apakah output sudah siap dipakai Streamlit?** Ya -- `dashboard_topic_data.csv` (ditambah
`topic_summary_*_final.csv` dan `topic_trends_final.csv` untuk ringkasan/tren) berisi seluruh
kolom yang dibutuhkan (`document_id`, `topic`, `topic_label_auto`, `topic_probability`, `text`,
`tahun`/`bulan`/`tahun_bulan`, metadata sumber) dan **tidak membutuhkan SBERT/BERTopic apa pun
untuk dibaca** -- dashboard hanya perlu `pandas.read_csv()`. `reviewer_name` dan alamat lengkap
sudah dipastikan tidak ada di file ini (lihat `assert` pada Bagian 11).

**Yang perlu diingat saat membangun dashboard (di luar notebook ini):**
- `topic_label_auto` adalah interpretasi **otomatis**, bukan ground truth -- pertimbangkan
  menampilkan top_words di dashboard juga agar pengguna dashboard bisa menilai sendiri.
- Topik `-1` ("Outlier / Tidak Terklasifikasi") sebaiknya ditampilkan terpisah dari topik
  reguler di dashboard, bukan digabung ke salah satu topik substantif.
- Semua angka di `final_topic_config.json` dan checklist di atas bisa dikutip langsung untuk
  bagian metodologi/hasil skripsi (Bab III/IV).
